## Log and register model: native PyTorch

### Purpose
This notebook ingests a native PyTorch model trained outside Databricks,
logs it to MLflow using the native `mlflow.pytorch` flavour, and registers it in Unity Catalog.

Use this notebook when:
- The customer provides a serialized PyTorch model (`torch_model.pt`) and `artifacts.json` for metadata
- No custom PyFunc wrapper is required; you want the native PyTorch flavour for deployment
- The model class (e.g. SimpleClassifierNet) must match what was used in export/training

This is Stage 2 of the BYOM workflow (Log & Register).

### Expected inputs
Artifacts must already exist in a Unity Catalog volume and follow the expected layout defined in Stage 1.

Required:
- `torch_model.pt` (serialized nn.Module or bundle with model, scaler, meta)
- `artifacts.json` (feature columns, example input, etc.)

Optional but recommended:
- `checksums.json` for artifact integrity validation


In [ ]:
%pip install torch
%pip install --upgrade 'mlflow>=3.0.0'
%restart_python

In [ ]:
dbutils.widgets.text("catalog_name", "pcl", "Catalog Name")
dbutils.widgets.text("schema_name", "byo_model", "Schema Name")
dbutils.widgets.text("volume_name", "volume", "Volume Name")

catalog = dbutils.widgets.get("catalog_name")
schema = dbutils.widgets.get("schema_name")
volume = dbutils.widgets.get("volume_name")

### Load Artifacts from Unity Catalog Volume

Reads model artifacts from the configured UC volume using notebook widgets:

- `catalog_name`
- `schema_name`
- `volume_name`

Ensure these are aligned before running.

If `checksums.json` is present:
- Verify artifact integrity before logging
- Fail fast if mismatches are detected

Recommended for production engagements.


In [ ]:
%run ./00_shared

In [ ]:
import torch
import torch.nn as nn
import pandas as pd
import numpy as np
import mlflow
from mlflow.tracking import MlflowClient

# Must match the class used in 0_export_model_and_artifacts; required for torch.load to unpickle torch_model.pt
class SimpleClassifierNet(nn.Module):
    def __init__(self, in_dim, hid_dim, out_dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_dim, hid_dim), nn.ReLU(),
            nn.Linear(hid_dim, out_dim),
        )
    def forward(self, x):
        return self.net(x)

# --- Paths and verification ---
model_name = "native_torch"
artifact_root = get_artifact_root(catalog, schema, volume)
dl_config = get_dl_config(artifact_root)
checksums_path = get_checksums_path(artifact_root)
verify_artifacts(checksums_path, [("artifacts.json", dl_config["artifacts_json_path"]), ("torch_model.pt", dl_config["torch_model_path"])])
canonical_input, feature_columns = load_canonical_input_from_artifacts_json(dl_config["artifacts_json_path"])


### Infer model signature

Infers input/output schema using canonical sample input (if provided).  
The inferred signature ensures consistent contract enforcement during serving and batch inference.

In [ ]:
# --- Load torch model (and optional scaler), run one prediction, infer MLflow signature ---
torch_model, scaler = load_torch_artifact(dl_config["torch_model_path"])
if isinstance(torch_model, nn.Module):
    torch_model.eval()
X = canonical_input[feature_columns].to_numpy(dtype=np.float32)
if scaler is not None:
    X = scaler.transform(X)
with torch.no_grad():
    logits = torch_model(torch.from_numpy(X.astype(np.float32)))
    probs = torch.softmax(logits, dim=1).numpy()
    pred_class = probs.argmax(axis=1)
y_pred = pd.DataFrame({"prediction": pred_class.astype(np.float64)})
signature = validate_and_infer_signature(canonical_input, y_pred)

### Log to MLflow and register to Unity Catalog

Logs the native PyTorch model using `mlflow.pytorch`. Artifacts, signature, and metadata are recorded in MLflow tracking.

Registers the logged model in Unity Catalog Model Registry.

Ensures:
- Governance
- Lineage
- Versioning

In [ ]:
# --- Log to MLflow using native PyTorch flavor (model only; no scaler in saved artifact), register in UC ---
mlflow.set_registry_uri("databricks-uc")
deps = ["torch", "pandas", "numpy", "scikit-learn", "joblib"]
input_example_np = canonical_input[feature_columns].to_numpy(dtype=np.float32)
if scaler is not None:
    input_example_np = scaler.transform(input_example_np)
with mlflow.start_run() as run:
    mlflow.pytorch.log_model(torch_model, artifact_path=model_name, input_example=input_example_np, signature=signature, extra_pip_requirements=deps)
    model_uri = f"runs:/{run.info.run_id}/{model_name}"
registered_name = f"{catalog}.{schema}.{model_name}"
result = mlflow.register_model(model_uri=model_uri, name=registered_name)
client = MlflowClient()
register_in_uc(client, model_uri, registered_name, str(result.version))

# For job pipelines: pass version to downstream tasks (e.g. evaluation)
dbutils.jobs.taskValues.set(key="model_version", value=str(result.version))
dbutils.jobs.taskValues.set(key="model_name", value=registered_name)

# Sanity check: load the version we just registered and run predict (Champion is set later in 3_model_approval)
loaded = mlflow.pyfunc.load_model(f"models:/{registered_name}/{result.version}")
loaded.predict(canonical_input)
print(f"{registered_name} v{result.version} loaded; predict OK.")